In [4]:
"""
Sales Prediction using Advertising Data
========================================
Predicts future sales based on TV, Radio, and Newspaper advertising spend.
Models used: Linear Regression, Ridge, Lasso, Random Forest
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import joblib
import os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

os.makedirs('outputs', exist_ok=True)
os.makedirs('models', exist_ok=True)

# ──────────────────────────────────────────────
# 1. LOAD & EXPLORE DATA
# ──────────────────────────────────────────────
print("=" * 60)
print("  SALES PREDICTION — ADVERTISING DATASET")
print("=" * 60)

df = pd.read_csv('Advertising.csv')
df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')

print("\n📋 Dataset Shape:", df.shape)
print("\n📊 First 5 rows:\n", df.head())
print("\n📈 Statistical Summary:\n", df.describe().round(2))
print("\n✅ Missing Values:\n", df.isnull().sum())

# ──────────────────────────────────────────────
# 2. DATA CLEANING & FEATURE ENGINEERING
# ──────────────────────────────────────────────
print("\n🔧 Feature Engineering...")

# Total spend feature
df['Total_Ad_Spend'] = df['TV'] + df['Radio'] + df['Newspaper']

# TV dominance ratio
df['TV_Ratio'] = df['TV'] / df['Total_Ad_Spend']
df['Radio_Ratio'] = df['Radio'] / df['Total_Ad_Spend']

# Interaction: TV × Radio synergy
df['TV_Radio_Interaction'] = df['TV'] * df['Radio']

# Budget tiers
df['Budget_Tier'] = pd.cut(
    df['Total_Ad_Spend'],
    bins=[0, 100, 200, 350, 500],
    labels=['Low', 'Medium', 'High', 'Very High']
)

print("   Added: Total_Ad_Spend, TV_Ratio, Radio_Ratio, TV_Radio_Interaction, Budget_Tier")
print(f"   Final shape: {df.shape}")

# ──────────────────────────────────────────────
# 3. EXPLORATORY DATA ANALYSIS (PLOTS)
# ──────────────────────────────────────────────
print("\n📊 Generating EDA plots...")

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Advertising Data — Exploratory Analysis', fontsize=16, fontweight='bold', y=1.01)

channels = ['TV', 'Radio', 'Newspaper']
for i, ch in enumerate(channels):
    axes[0, i].scatter(df[ch], df['Sales'], alpha=0.6, color=COLORS[i], edgecolors='white', s=50)
    z = np.polyfit(df[ch], df['Sales'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[ch].min(), df[ch].max(), 100)
    axes[0, i].plot(x_line, p(x_line), color='black', linewidth=2, linestyle='--')
    axes[0, i].set_xlabel(f'{ch} Ad Spend ($K)', fontsize=11)
    axes[0, i].set_ylabel('Sales (units)', fontsize=11)
    axes[0, i].set_title(f'{ch} vs Sales  (r={df[ch].corr(df["Sales"]):.2f})', fontsize=12)

# Sales distribution
axes[1, 0].hist(df['Sales'], bins=20, color=COLORS[0], edgecolor='white', alpha=0.85)
axes[1, 0].axvline(df['Sales'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean={df["Sales"].mean():.1f}')
axes[1, 0].set_xlabel('Sales (units)', fontsize=11)
axes[1, 0].set_ylabel('Frequency', fontsize=11)
axes[1, 0].set_title('Sales Distribution', fontsize=12)
axes[1, 0].legend()

# Correlation heatmap
corr = df[['TV', 'Radio', 'Newspaper', 'Total_Ad_Spend', 'Sales']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1, 1],
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
axes[1, 1].set_title('Correlation Matrix', fontsize=12)

# Sales by Budget Tier
tier_stats = df.groupby('Budget_Tier')['Sales'].mean()
bars = axes[1, 2].bar(tier_stats.index, tier_stats.values, color=COLORS[:4], edgecolor='white', width=0.6)
axes[1, 2].set_xlabel('Budget Tier', fontsize=11)
axes[1, 2].set_ylabel('Avg Sales (units)', fontsize=11)
axes[1, 2].set_title('Avg Sales by Budget Tier', fontsize=12)
for bar, val in zip(bars, tier_stats.values):
    axes[1, 2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                    f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/01_eda_analysis.png', dpi=150, bbox_inches='tight')
plt.close()
print("   Saved: outputs/01_eda_analysis.png")

# ──────────────────────────────────────────────
# 4. MODEL PREPARATION
# ──────────────────────────────────────────────
print("\n🤖 Preparing models...")

features_basic = ['TV', 'Radio', 'Newspaper']
features_full  = ['TV', 'Radio', 'Newspaper', 'Total_Ad_Spend', 'TV_Radio_Interaction']

X = df[features_full]
y = df['Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

joblib.dump(scaler, 'models/scaler.pkl')

# ──────────────────────────────────────────────
# 5. TRAIN & EVALUATE MODELS
# ──────────────────────────────────────────────
print("\n🏋️  Training models...")

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression' : Ridge(alpha=1.0),
    'Lasso Regression' : Lasso(alpha=0.1),
    'Random Forest'    : RandomForestRegressor(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    X_tr = X_train_sc if name != 'Random Forest' else X_train
    X_te = X_test_sc  if name != 'Random Forest' else X_test
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    cv_scores = cross_val_score(model, X_tr, y_train, cv=5, scoring='r2')
    results[name] = {
        'model'   : model,
        'y_pred'  : y_pred,
        'R2'      : r2_score(y_test, y_pred),
        'MAE'     : mean_absolute_error(y_test, y_pred),
        'RMSE'    : np.sqrt(mean_squared_error(y_test, y_pred)),
        'CV_R2'   : cv_scores.mean(),
        'CV_Std'  : cv_scores.std(),
    }
    joblib.dump(model, f'models/{name.replace(" ", "_").lower()}.pkl')

print("\n📊 Model Performance:")
print(f"{'Model':<22} {'R²':>6} {'CV R²':>8} {'MAE':>7} {'RMSE':>7}")
print("-" * 56)
for name, r in results.items():
    print(f"{name:<22} {r['R2']:>6.4f} {r['CV_R2']:>8.4f} {r['MAE']:>7.3f} {r['RMSE']:>7.3f}")

# ──────────────────────────────────────────────
# 6. MODEL COMPARISON PLOT
# ──────────────────────────────────────────────
print("\n📊 Generating model comparison plots...")

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')

model_names = list(results.keys())

# R² bar chart
r2_vals = [results[m]['R2'] for m in model_names]
bars = axes[0, 0].bar(model_names, r2_vals, color=COLORS, edgecolor='white', width=0.5)
axes[0, 0].set_ylim(0, 1.05)
axes[0, 0].set_title('R² Score (higher = better)', fontsize=12)
axes[0, 0].set_ylabel('R²')
axes[0, 0].tick_params(axis='x', rotation=15)
for bar, val in zip(bars, r2_vals):
    axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')

# RMSE bar chart
rmse_vals = [results[m]['RMSE'] for m in model_names]
bars2 = axes[0, 1].bar(model_names, rmse_vals, color=COLORS, edgecolor='white', width=0.5)
axes[0, 1].set_title('RMSE (lower = better)', fontsize=12)
axes[0, 1].set_ylabel('RMSE')
axes[0, 1].tick_params(axis='x', rotation=15)
for bar, val in zip(bars2, rmse_vals):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

# Best model: Actual vs Predicted
best_name = max(results, key=lambda m: results[m]['R2'])
best = results[best_name]
axes[1, 0].scatter(y_test, best['y_pred'], alpha=0.7, color=COLORS[0], edgecolors='white', s=60)
mn, mx = y_test.min(), y_test.max()
axes[1, 0].plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect prediction')
axes[1, 0].set_xlabel('Actual Sales', fontsize=11)
axes[1, 0].set_ylabel('Predicted Sales', fontsize=11)
axes[1, 0].set_title(f'{best_name} — Actual vs Predicted\nR²={best["R2"]:.4f}', fontsize=12)
axes[1, 0].legend()

# Residuals
residuals = y_test.values - best['y_pred']
axes[1, 1].scatter(best['y_pred'], residuals, alpha=0.7, color=COLORS[2], edgecolors='white', s=60)
axes[1, 1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Predicted Sales', fontsize=11)
axes[1, 1].set_ylabel('Residuals', fontsize=11)
axes[1, 1].set_title('Residuals Plot (Best Model)', fontsize=12)

plt.tight_layout()
plt.savefig('outputs/02_model_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("   Saved: outputs/02_model_comparison.png")

# ──────────────────────────────────────────────
# 7. FEATURE IMPORTANCE & IMPACT ANALYSIS
# ──────────────────────────────────────────────
print("\n📊 Generating impact analysis plots...")

rf_model = results['Random Forest']['model']
lr_model = results['Linear Regression']['model']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Advertising Impact Analysis', fontsize=14, fontweight='bold')

# Feature importance (RF)
importances = pd.Series(rf_model.feature_importances_, index=features_full).sort_values(ascending=True)
importances.plot(kind='barh', ax=axes[0], color=COLORS[0], edgecolor='white')
axes[0].set_title('Feature Importance (Random Forest)', fontsize=12)
axes[0].set_xlabel('Importance Score')

# LR coefficients
coefs = pd.Series(lr_model.coef_, index=features_full).sort_values(ascending=True)
colors_coef = ['#C73E1D' if c < 0 else '#2E86AB' for c in coefs.values]
coefs.plot(kind='barh', ax=axes[1], color=colors_coef, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('Feature Coefficients (Linear Regression)', fontsize=12)
axes[1].set_xlabel('Coefficient Value')

# Budget impact simulation
tv_range = np.linspace(0, 300, 100)
sim_data = pd.DataFrame({
    'TV': tv_range,
    'Radio': [df['Radio'].mean()] * 100,
    'Newspaper': [df['Newspaper'].mean()] * 100,
    'Total_Ad_Spend': tv_range + df['Radio'].mean() + df['Newspaper'].mean(),
    'TV_Radio_Interaction': tv_range * df['Radio'].mean(),
})
sim_pred = lr_model.predict(scaler.transform(sim_data[features_full]))
axes[2].plot(tv_range, sim_pred, color=COLORS[0], linewidth=3, label='Predicted Sales')
axes[2].axvline(df['TV'].mean(), color='gray', linestyle='--', linewidth=1.5, label=f'Avg TV spend (${df["TV"].mean():.0f}K)')
axes[2].fill_between(tv_range, sim_pred, alpha=0.15, color=COLORS[0])
axes[2].set_xlabel('TV Ad Spend ($K)', fontsize=11)
axes[2].set_ylabel('Predicted Sales (units)', fontsize=11)
axes[2].set_title('Sales vs TV Budget (Simulated)', fontsize=12)
axes[2].legend()

plt.tight_layout()
plt.savefig('outputs/03_impact_analysis.png', dpi=150, bbox_inches='tight')
plt.close()
print("   Saved: outputs/03_impact_analysis.png")

# ──────────────────────────────────────────────
# 8. ACTIONABLE INSIGHTS
# ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("  📌 ACTIONABLE BUSINESS INSIGHTS")
print("=" * 60)

tv_coef   = lr_model.coef_[features_full.index('TV')]
rad_coef  = lr_model.coef_[features_full.index('Radio')]
news_coef = lr_model.coef_[features_full.index('Newspaper')]

print(f"""
1. 📺 TV ADVERTISING has the HIGHEST impact.
   → Every $1K increase in TV spend ≈ +{tv_coef:.2f} units in sales.
   → TV correlation with sales: {df['TV'].corr(df['Sales']):.2f}

2. 📻 RADIO ADVERTISING is the 2nd most effective channel.
   → Every $1K increase in Radio spend ≈ +{rad_coef:.2f} units in sales.
   → High ROI per dollar spent vs TV.

3. 📰 NEWSPAPER ADVERTISING shows LOWEST impact.
   → Newspaper correlation: {df['Newspaper'].corr(df['Sales']):.2f}
   → Consider reallocating newspaper budget to TV or Radio.

4. 🔗 TV + RADIO COMBINATION is the most powerful strategy.
   → Their interaction term boosts prediction accuracy significantly.
   → Combined campaigns outperform single-channel ones.

5. 🏆 BEST MODEL: {best_name}
   → R² = {best['R2']:.4f} | RMSE = {best['RMSE']:.3f} | MAE = {best['MAE']:.3f}

6. 💡 BUDGET RECOMMENDATIONS:
   → Low budget  → Prioritise Radio (best ROI per $).
   → High budget → Lead with TV, supplement with Radio.
   → Reduce Newspaper allocation for underperforming markets.
""")

# ──────────────────────────────────────────────
# 9. SAVE PREDICTIONS
# ──────────────────────────────────────────────
X_all_sc = scaler.transform(df[features_full])
df['Predicted_Sales'] = rf_model.predict(df[features_full])
df.to_csv('outputs/predictions_with_actuals.csv', index=False)
print("✅ Predictions saved to: outputs/predictions_with_actuals.csv")
print("✅ Models saved to:      models/")
print("\nAll done! 🎉")

  SALES PREDICTION — ADVERTISING DATASET

📋 Dataset Shape: (200, 4)

📊 First 5 rows:
       TV  Radio  Newspaper  Sales
0  230.1   37.8       69.2   22.1
1   44.5   39.3       45.1   10.4
2   17.2   45.9       69.3    9.3
3  151.5   41.3       58.5   18.5
4  180.8   10.8       58.4   12.9

📈 Statistical Summary:
            TV   Radio  Newspaper   Sales
count  200.00  200.00     200.00  200.00
mean   147.04   23.26      30.55   14.02
std     85.85   14.85      21.78    5.22
min      0.70    0.00       0.30    1.60
25%     74.38    9.98      12.75   10.38
50%    149.75   22.90      25.75   12.90
75%    218.82   36.52      45.10   17.40
max    296.40   49.60     114.00   27.00

✅ Missing Values:
 TV           0
Radio        0
Newspaper    0
Sales        0
dtype: int64

🔧 Feature Engineering...
   Added: Total_Ad_Spend, TV_Ratio, Radio_Ratio, TV_Radio_Interaction, Budget_Tier
   Final shape: (200, 9)

📊 Generating EDA plots...
   Saved: outputs/01_eda_analysis.png

🤖 Preparing models...

